# Analyse exploratoire – Prix des denrées sur les marchés du Bénin

On part de `data/prix_propres.parquet`, créé à la fin de `01_cleaning.ipynb`. Le but est de comprendre comment les prix bougent (tendances, saisons, écarts entre marchés) et de voir quelles informations aideront à prévoir le prix du mois suivant.

Le notebook suit trois parties :
1. Analyse univariée
2. Analyse bivariée
3. Analyse multivariée

Ici, on ne modifie pas les données. Si un problème de qualité apparaît, on le note et on le corrige dans `01_cleaning.ipynb`.

## 0.1 Chargement

On charge le fichier propre et on trie par date, sinon les courbes feront des allers-retours. On ajoute le nom français des produits (pour les figures) et la colonne `mois`. Ces ajouts restent dans le notebook : le fichier n'est pas modifié.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_parquet("../data/prix_propres.parquet")
df = df.sort_values(["commodity", "market", "date"]).reset_index(drop=True)

noms = {
    "Cassava meal (gari)": "Gari",
    "Rice (imported)": "Riz importé",
    "Maize (white)": "Maïs blanc",
    "Beans (white)": "Haricot blanc",
    "Tomatoes": "Tomate",
    "Oil (groundnut)": "Huile d'arachide",
    "Onions": "Oignon",
}
df["produit"] = df["commodity"].map(noms)
df["mois"] = df["date"].dt.to_period("M")
produits = list(noms.values())

os.makedirs("../reports/figures", exist_ok=True)
print(df.shape)
df.head()

**Conclusion.** On a 21 394 relevés, 313 séries (un produit sur un marché) et 7 produits, de janvier 2019 à février 2026. L'huile est au litre, les autres produits au kilo.

# Partie 1 – Analyse univariée

On regarde chaque variable seule, pour connaître l'ordre de grandeur des prix et de leurs variations, et repérer les trous.

## 1.1 Comment les prix sont-ils répartis pour chaque produit ?

D'abord un tableau avec les chiffres de chaque produit. L'asymétrie dit si la courbe est penchée : proche de 0, les prix sont répartis de façon équilibrée autour de la médiane. Au-dessus de 1, il y a une longue queue de prix très élevés. On calcule aussi l'asymétrie du log du prix, pour voir si passer au log arrangerait les choses.

In [ ]:
import numpy as np

tableau = df.groupby("produit")["price"].describe().round(0).loc[produits]
tableau["asymétrie"] = df.groupby("produit")["price"].skew().round(2)
tableau["asymétrie (log)"] = np.log(df["price"]).groupby(df["produit"]).skew().round(2)
tableau

**Ce que dit le tableau.** La tomate (asymétrie 1,92) et l'oignon (2,87) ont une longue queue de prix très élevés : leur maximum vaut 8 à 9 fois leur médiane (3 300 et 3 972 FCFA/kg, pour des médianes de 409 et 428). Le riz (1,34) et le haricot (0,94) sont un peu penchés. Le gari, le maïs et l'huile (0,18 à 0,63) sont presque équilibrés. Le log aide surtout pour la tomate (1,92 → -0,24), l'oignon (2,87 → -0,11) et le riz (1,34 → 0,47), mais il ne sert à rien pour le gari, le maïs et l'huile.

Maintenant les histogrammes, un par produit avec sa propre échelle, puisque l'huile est autour de 1 150 FCFA et le maïs autour de 200.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.flatten()

for ax, p in zip(axes, produits):
    ax.hist(df[df["produit"] == p]["price"], bins=40, color="steelblue")
    ax.set_title(p)
    ax.set_xlabel("Prix (FCFA/L)" if p == "Huile d'arachide" else "Prix (FCFA/kg)")

axes[7].axis("off")
plt.tight_layout()
plt.savefig("../reports/figures/1_1_distribution_prix.png", dpi=150)
plt.show()

**Ce que dit la figure.** Les prix du gari, du maïs, du riz, du haricot et de l'huile forment une cloche resserrée autour de leur médiane : 99 % des prix restent sous 2 fois la médiane (1,5 fois pour l'huile). La tomate et l'oignon s'étalent beaucoup plus, avec une queue qui part très loin à droite (99 % sous 3,3 et 3,2 fois la médiane, mais un maximum à 8 et 9 fois). On travaillera donc sur le log du prix, au moins pour la tomate et l'oignon. Le maximum de l'oignon (3 972 FCFA/kg, Natitingou, juillet 2025) reste à vérifier.

## 1.2 De combien le prix change-t-il d'un mois à l'autre ?

Il faut d'abord remettre chaque série sur un calendrier mensuel complet (86 mois, de janvier 2019 à février 2026). Sinon, un écart de trois mois serait compté comme une variation d'un mois. On met donc les séries en colonnes, une ligne par mois, et un mois sans relevé devient une case vide. La variation en % est vide dès qu'un des deux mois est vide.

In [ ]:
prix_large = df.pivot_table(index="mois", columns=["commodity", "market"], values="price")
tous_mois = pd.period_range("2019-01", "2026-02", freq="M")
prix_large = prix_large.reindex(tous_mois)

variation = prix_large.pct_change(fill_method=None) * 100

var_long = variation.stack(["commodity", "market"]).dropna().reset_index()
var_long.columns = ["mois", "commodity", "market", "variation"]
var_long["produit"] = var_long["commodity"].map(noms)

print(prix_large.shape)
print("cases vides :", prix_large.isna().sum().sum())
print("variations calculées :", len(var_long))

**Conclusion.** Sur le calendrier complet, 5 524 cases sont vides sur 26 918 (environ 20 %). Cela vient des séries qui commencent tard et des mois manquants. On peut calculer 18 074 variations sur 21 394 relevés : on en perd 313 (le premier mois de chaque série) et environ 3 000 à cause des trous.

Le tableau : `describe` de la variation par produit, la moyenne des variations sans le signe (l'erreur qu'on ferait en disant « le prix ne change pas »), et la part des mois où le prix bouge de plus de 10 %.

In [ ]:
resume = var_long.groupby("produit")["variation"].describe().round(1).loc[produits]
absolue = var_long["variation"].abs()
resume["variation moyenne sans signe"] = absolue.groupby(var_long["produit"]).mean().round(1)
resume["part > 10 % (en %)"] = (absolue > 10).groupby(var_long["produit"]).mean().mul(100).round(0)
resume

**Ce que dit le tableau.** Le prix change peu d'un mois à l'autre pour le riz (3,2 % en moyenne sans le signe, 10 % des mois à plus de 10 %) et pour l'huile (4,5 % ; 13 %). Il bouge beaucoup plus pour le gari, le haricot et le maïs (environ 8 % ; 27 à 33 %), et énormément pour l'oignon (20,9 % ; 59 %) et la tomate (27,9 % ; 69 %). La médiane est à 0 pour presque tous : très souvent, le prix ne change pas. Le modèle « le prix ne change pas » se trompera donc d'environ 3 % sur le riz, mais de 20 à 28 % sur l'oignon et la tomate.

Les histogrammes de la variation, un par produit. Une poignée de variations sont énormes (jusqu'à +1 200 % pour la tomate), alors on limite l'axe à ±60 % pour voir le centre.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.flatten()

for ax, p in zip(axes, produits):
    ax.hist(var_long[var_long["produit"] == p]["variation"], bins=40, range=(-60, 60), color="steelblue")
    ax.set_title(p)
    ax.set_xlabel("Variation mensuelle (%)")

axes[7].axis("off")
plt.tight_layout()
plt.savefig("../reports/figures/1_2_variation_mensuelle.png", dpi=150)
plt.show()

**Ce que dit la figure.** Tous les produits ont un grand pic à 0 : c'est le prix qui reste identique d'un mois à l'autre. Le riz et l'huile sont très resserrés autour de 0, alors que la tomate et l'oignon forment une courbe large qui s'étale sur toute la plage. Sur 18 074 variations, 416 dépassent ±60 % (269 pour la tomate, 123 pour l'oignon), et ce sont presque les seules. À garder en tête pour l'erreur du modèle sur ces deux produits.

## 1.3 Combien de séries sont relevées chaque mois ?

Sur le calendrier complet, on compte combien de séries ont un prix chaque mois. Les creux de la courbe seront des mois où il manquera des prix passés.

In [ ]:
par_mois = prix_large.notna().sum(axis=1)
par_mois.index = par_mois.index.to_timestamp()

plt.figure(figsize=(12, 4))
plt.plot(par_mois)
plt.ylabel("Nombre de séries relevées")
plt.title("Nombre de séries présentes chaque mois")
plt.savefig("../reports/figures/1_3_series_par_mois.png", dpi=150)
plt.show()

par_mois[par_mois < 250]

**Conclusion.** Depuis 2020, on a environ 270 séries par mois sur 313 (maximum 284). Les trous sont tous au début ou isolés : 110 séries début 2019, seulement 5 en mai et juin 2019, 76 en novembre 2019, et **aucun relevé en mars 2022**. Sur 86 mois, 13 sont sous 250 séries. À cause de mars 2022, les décalages de 1, 2 et 3 mois manquent pour avril, mai et juin 2022, et celui de 12 mois pour mars 2023. Il faudra aussi se méfier de 2019, trop pauvre.

## 1.4 Les marchés couvrent-ils bien le pays ?

On compte les marchés et les séries de chaque département (`admin1`), du nord au sud avec la latitude moyenne.

In [ ]:
marches = df.drop_duplicates("market")
dep = marches.groupby("admin1").agg(marches=("market", "nunique"), latitude=("latitude", "mean")).round(2)

series = df.drop_duplicates(["commodity", "market"])
dep["séries"] = series.groupby("admin1").size()

dep.sort_values("latitude", ascending=False)

**Conclusion.** Les 12 départements sont présents (50 marchés, 313 séries). Le nord (Alibori, Atakora, Borgou, Donga) compte 17 marchés et 106 séries, le centre et le sud 33 marchés et 207 séries : on peut comparer le nord et le sud. Par contre, le Mono et le Littoral n'ont qu'un marché chacun (5 et 7 séries), donc leur « département » n'est que ce marché.

## Bilan de la partie 1

- Les prix du gari, du maïs, du riz, du haricot et de l'huile sont équilibrés autour de leur médiane. La tomate et l'oignon ont une longue queue de prix élevés : on passera au log pour eux.
- Le prix change peu d'un mois à l'autre pour le riz (3 % en moyenne) et l'huile (4,5 %), beaucoup pour la tomate (28 %) et l'oignon (21 %). Le modèle naïf se trompera surtout sur ces deux produits.
- Environ 35 % des variations mensuelles dépassent 10 %, et 69 % pour la tomate.
- Le calendrier complet a 20 % de cases vides : les décalages devront accepter des valeurs manquantes.
- Aucun relevé en mars 2022, et très peu en 2019 : les décalages sont impossibles autour de mars 2022, et on se méfiera de 2019.
- Les 12 départements sont couverts, mais le Mono et le Littoral n'ont qu'un marché chacun.
- À vérifier : le maximum de l'oignon (3 972 FCFA/kg) et les variations extrêmes de la tomate et de l'oignon.